> **Código académico, sin garantía.** Material de los cursos de Modelado y Diseño de Líneas de Transmisión (Universidad Nacional de Colombia). Se entrega tal cual, sin garantía de ninguna clase; **no debe usarse para decisiones de diseño, operación o seguridad de instalaciones reales** sin verificación independiente. Ver [`DISCLAIMER.md`](../../DISCLAIMER.md). Licencia por definir (intención: código abierto).

# Corona en una línea aérea: 
## Objetivo
El efecto Corona es un fenómeno importante de analizar en las líneas de alta tensión, sus efectos ocasionan pérdidas, ruido y radiointerferencia.  Este fenómeno se da por las altas concetraciones de campo eléctrico en ciertas partes del sistema.  Para evitar el efecto corona debemos disminuir el gradiente elétrico en las zonas más críticas, es decir controlar el campo eléctrico.

## Notebook: Gradiente del haz, límite de Peek y comparación con CIGRE
Este notebook calcula el gradiente en la superficie de los conductores de una línea trifásica, con lo que se puede observar el gradiente de campo eléctrico máximo en los conductores, asi como comparar con las ecuaciones de gradiente crítico de Peek y con las ecuaciones de CIGRE, Adicionalmetne se puede estimar el ruido audible y radiointerferencia.

**Cómo usarlo.** Cambia los valores de la sección 1 y ejecuta todo (*Run All*). Los diagramas de la sección 2 se redibujan con tus números, así compruebas que la geometría es la que querías antes de mirar resultados.

| Sección | Qué hace |
|---|---|
| 1 | Parámetros que se pueden variar |
| 2 | Diagrama de la geometría y de los parámetros |
| 3 | Gradiente en la superficie y margen contra Peek |
| 4 | Recorrido por la superficie de un subconductor, con barra deslizante |
| 5 | Gradiente en el tiempo durante un ciclo |
| 6 | Acercamiento a un haz: campo y líneas equipotenciales |
| 7 | Animación de un ciclo de 60 Hz |
| 8 | Campo eléctrico al nivel del suelo |
| 9 | Comparación con las ecuaciones de Kuffel y de Peek |
| 10 | Comparación con el libro de CIGRE (Nolasco et al.): caso base y figura 4.62 |
| 11 | Variaciones y datos para el taller: barridos, exportar a CSV, cargar tu geometría |

El taller para los estudiantes está en `Corona_taller.ipynb`; su enunciado es un documento Word que se entrega aparte (`Taller_corona.docx`).


## Dos tipos de cálculo: simulación de cargas y fórmulas

Los resultados de este notebook salen de dos tipos de cálculo. Cada tabla y cada figura dice cuál se usó: `(simulación)` o `simulación de cargas` para el primero; `(Kuffel)`, `(Peek)` o `fórmula` para el segundo.

| | Simulación de cargas | Fórmulas |
|---|---|---|
| Qué es | Método numérico. Cada conductor se reemplaza por varias **cargas lineales ficticias** ubicadas dentro de él. Se calcula su valor para que la superficie del conductor quede a su tensión (y los cables de guarda a tierra, a 0 V). Con esas cargas se obtiene el campo en cualquier punto. | Ecuaciones cerradas de la literatura, que se evalúan directamente. |
| Qué entrega aquí | El gradiente máximo `E_max` en la superficie de cada subconductor, el campo al nivel del suelo y el campo y las equipotenciales de las figuras y animaciones. | El gradiente crítico `E_c` (Peek), un valor aproximado de `E_max` (Kuffel / CIGRE, a partir de la carga de la fase) y las ecuaciones del libro de CIGRE. |
| Exactitud | Usa la geometría exacta: todas las fases, los guardas y la tierra. | Hacen simplificaciones, por ejemplo ignoran el campo de las otras fases. |

Las cargas ficticias (`N_CHARGES` por conductor) son una herramienta del método y no son cargas reales de la línea. La carga real de una fase, por unidad de longitud, es la suma de las cargas ficticias de todos sus subconductores.

In [ ]:
# Esta sección de código se encarga de configurar el entorno para ejecutar un cuaderno de 
# Jupyter que calcula campos eléctricos alrededor de cables de transmisión con el método de simulación de cargas.

import pathlib
import sys

# Locate the repository root (the folder that contains src/) so the notebook runs from anywhere.
root = pathlib.Path.cwd().resolve()
while not (root / "src" / "electric_field").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src" / "electric_field"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import corona
import importlib
import charge_simulation

# Recarga los módulos para usar siempre la versión actual del código, aunque el kernel ya los hubiera importado.
importlib.reload(corona)
importlib.reload(charge_simulation)
from charge_simulation import (Wire, add_bundle, charge_per_group, plot_field_map, plot_surface_gradient,
                               solve, three_phase_potentials, V_PER_M_TO_KV_PER_CM, PALETTE,
                               draw_bundle_zoom, draw_field_map_at)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

from matplotlib import animation
from IPython.display import HTML, display


# Esta función lee la geometría de una línea desde un archivo CSV con las columnas kind, x, y.
# kind es "phase" (las tres fases, en el orden a, b, c) o "guard" (cables de guarda). x e y en metros.
def load_geometry(path):
    """Return (phases, guards) as lists of (x, y) read from a CSV file."""
    df = pd.read_csv(path)
    phases = [(float(r.x), float(r.y)) for r in df[df["kind"] == "phase"].itertuples()]
    guards = [(float(r.x), float(r.y)) for r in df[df["kind"] == "guard"].itertuples()]
    if len(phases) != 3:
        raise ValueError(f"the file must have exactly 3 rows with kind = phase, it has {len(phases)}")
    return phases, guards


# Esta función guarda la geometría actual en un CSV, como plantilla para editarla en Excel.
def save_geometry(path, phases, guards):
    rows = [("phase", x, y) for x, y in phases] + [("guard", x, y) for x, y in guards]
    pd.DataFrame(rows, columns=["kind", "x", "y"]).to_csv(path, index=False)

## 1. Parámetros

Todo lo que se puede variar está en la siguiente celda. Las unidades van en los comentarios.

| Parámetro | Significado | Efecto esperado |
|---|---|---|
| `V_LL_KV` | tensión de línea, valor eficaz (RMS) | el gradiente crece en proporción directa |
| `PHASES` | posición `(x, y)` del centro de cada haz, en el orden a, b, c | fases más separadas bajan el gradiente de la fase central |
| `N_SUB`, `SPACING`, `R_SUB` | subconductores por fase, separación entre contiguos y radio de uno | más subconductores o más separación bajan `E_max` |
| `GUARDS`, `R_GUARD`, `GUARD_MODE` | posición y radio de los cables de guarda, y su conexión | el guarda a tierra sube muy poco el gradiente de las fases |
| `ALTITUDE_M`, `TEMP_C` | sitio de la línea | más altura baja la densidad del aire y el gradiente crítico |
| `M_SURFACE` | factor de estado de la superficie de Peek | 0,8 cable trenzado seco; 0,3 a 0,6 con lluvia |
| `FREQ_HZ`, `PHASE_SEQUENCE` | frecuencia y orden de giro de las fases | la secuencia `"acb"` invierte el orden en que cada fase llega a su pico |
| `ZOOM_PHASE`, `ZOOM_WT_DEG`, `N_FRAMES`, `SAVE_GIF` | fase y instante del acercamiento, cuadros por ciclo y archivo GIF | solo cambian las figuras de las secciones 4, 6 y 7 |
| `TOUR_SUB` | subconductor del haz que se recorre en la sección 4 | 0 es el primero del haz |
| `OUTPUT_DIR` | carpeta donde la sección 11 guarda los CSV | |

In [ ]:
# ---- Line ------------------------------------------------------------------------------------
V_LL_KV = 500.0                                       # line-to-line rms voltage [kV]
PHASES = [(-11.0, 25.0), (0.0, 25.0), (11.0, 25.0)]   # (x, y) of the bundle centre of phases a, b, c [m]

# ---- Bundle (the same for the three phases) ---------------------------------------------------
N_SUB = 3          # sub-conductors per phase
SPACING = 0.45     # distance between adjacent sub-conductors [m]
R_SUB = 0.0141     # radius of ONE sub-conductor [m]

# ---- Shield wires -----------------------------------------------------------------------------
GUARDS = [(-7.0, 35.0), (7.0, 35.0)]   # (x, y) of each shield wire [m]; use [] for none
R_GUARD = 0.0055                       # shield wire radius [m]
GUARD_MODE = "grounded"                # "grounded" (0 V), "insulated" (floating) or "none"

# ---- Site and surface ---------------------------------------------------------------------------
ALTITUDE_M = 1500.0    # altitude above sea level [m]
TEMP_C = 20.0          # mean air temperature [C]
M_SURFACE = 0.8        # Peek surface factor (1.0 smooth, 0.8 dry stranded, 0.3-0.6 rain)

# ---- Electrical -------------------------------------------------------------------------------
FREQ_HZ = 60.0            # system frequency [Hz]
PHASE_SEQUENCE = "abc"    # rotation order of the phases: "abc" (positive) or "acb" (negative)

# ---- Zoom and animation -----------------------------------------------------------------------
ZOOM_PHASE = 1            # phase to zoom in on: 0 = a, 1 = b, 2 = c
ZOOM_WT_DEG = 120.0       # electrical angle of the zoom snapshot [deg]; with sequence abc phase b peaks at 120
N_FRAMES = 36             # frames per cycle in the animations (raise it for a smoother movie)
SAVE_GIF = None           # e.g. "corona_bundle.gif" also saves the bundle animation

# ---- Numerical --------------------------------------------------------------------------------
N_CHARGES = 12         # fictitious line charges per conductor in the charge simulation (raise it if the error is large)

# ---- Walk along the conductor surface (section 4) -------------------------------------------------
TOUR_SUB = 0              # sub-conductor of the zoomed phase to walk along (0 = first of the bundle)

# ---- Output (section 11) ----------------------------------------------------------------------------
OUTPUT_DIR = "corona_results"   # folder where export_results() writes the CSV files

# To use your own geometry from a CSV file (see section 11), uncomment:
# PHASES, GUARDS = load_geometry("my_line.csv")

## 2. Diagrama de la geometría

A la izquierda, la sección transversal de la línea con las cotas que definen `PHASES` y `GUARDS`. A la derecha, el haz de una fase con `SPACING`, `R_SUB` y el radio del haz `R = SPACING / (2 sen(π/n))`. Los conductores de la izquierda se dibujan más gruesos que en la realidad para que se vean; el haz de la derecha está a escala.

In [ ]:
# Esta función construye los cables y los potenciales de grupo para los parámetros definidos anteriormente.
def build_wires():
    """Wires and group potentials [V, complex peak] for the parameters above."""
    wires = []
    for i, (x, y) in enumerate(PHASES):
        add_bundle(wires, x, y, N_SUB, SPACING, R_SUB, group=i, label="abc"[i])
    pot = dict(enumerate(three_phase_potentials(V_LL_KV, PHASE_SEQUENCE)))
    if GUARD_MODE != "none":
        for j, (x, y) in enumerate(GUARDS):
            wires.append(Wire(x, y, R_GUARD, group=3 + j, label=f"G{j + 1}"))
            pot[3 + j] = 0.0 if GUARD_MODE == "grounded" else None
    return wires, pot

# Esta función dibuja las dimensiones y etiquetas en la figura de la sección transversal.
def dim(ax, p, q, text, off=(0, 0), color="0.25", ha="center"):
    """Double-headed dimension arrow from p to q with a label."""
    ax.annotate("", xy=q, xytext=p, arrowprops=dict(arrowstyle="<->", color=color, lw=1.2))
    ax.text((p[0] + q[0]) / 2 + off[0], (p[1] + q[1]) / 2 + off[1], text, color=color, ha=ha, va="center",
            fontsize=10, bbox=dict(fc="white", ec="none", pad=1.5))

# Esta función dibuja la figura de la sección transversal con las fases y los guardas, 
# así como un detalle del haz de subconductores a escala.
def draw_parameters():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={"width_ratios": [1.5, 1]})
    xs = [p[0] for p in PHASES] + [g[0] for g in GUARDS]
    ys = [p[1] for p in PHASES] + [g[1] for g in GUARDS]
    pad = 4.0
    a1.axhline(0, color="#8B7355", lw=3)
    a1.fill_between([min(xs) - pad, max(xs) + pad], -1.2, 0, color="#D9CBB0", alpha=0.6)
    for i, (x, y) in enumerate(PHASES):
        a1.add_patch(plt.Circle((x, y), 0.35, color=PALETTE[i], zorder=3))
        a1.text(x, y + 0.8, "abc"[i], color=PALETTE[i], ha="center", fontweight="bold", fontsize=13)
    for j, (x, y) in enumerate(GUARDS):
        a1.add_patch(plt.Circle((x, y), 0.25, color="#555555", zorder=3))
        a1.text(x, y + 0.8, f"G{j + 1}", color="#555555", ha="center", fontweight="bold", fontsize=12)
    xb, yb = PHASES[1]
    dim(a1, (xb + 0.9, 0), (xb + 0.9, yb), "y (altura del haz)", off=(1.9, 0))
    if GUARDS:
        yg = GUARDS[0][1]
        dim(a1, (min(xs) - 2.2, 0), (min(xs) - 2.2, yg), "y del guarda", off=(0, 0), color="#555555")
    dim(a1, (PHASES[0][0], PHASES[0][1] - 1.6), (PHASES[1][0], PHASES[1][1] - 1.6), "x entre fases", off=(0, -0.9))
    a1.set_xlim(min(xs) - pad, max(xs) + pad)
    a1.set_ylim(-1.2, max(ys) + 4)
    a1.set_aspect("equal")
    a1.set_xlabel("x [m]")
    a1.set_ylabel("Altura y [m]")
    a1.set_title("Congiguración Torre: PHASES y GUARDS")
    a1.grid(alpha=0.25)

    # bundle detail, to scale
    bw = add_bundle([], 0.0, 0.0, N_SUB, SPACING, R_SUB, group=0)
    R = corona.bundle_radius(N_SUB, SPACING)
    for w in bw:
        a2.add_patch(plt.Circle((w.x, w.y), w.r, color=PALETTE[1], zorder=3))
    a2.add_patch(plt.Circle((0, 0), R, fill=False, ls="--", color="0.5"))
    a2.plot(0, 0, "+", color="0.3")
    lim = R + 4 * R_SUB + 0.1
    if N_SUB >= 2:
        w0, w1 = bw[0], bw[1]
        dim(a2, (w0.x, w0.y), (w1.x, w1.y), f"SPACING = {SPACING * 100:.0f} cm", off=(0, 0.03 * lim / 0.3))
        a2.annotate("", xy=(bw[0].x, bw[0].y), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="0.3"))
        a2.text(bw[0].x / 2 - 0.02, bw[0].y / 2 - 0.03, f"R = {R * 100:.1f} cm", color="0.3", fontsize=10)
    w = bw[-1]
    a2.annotate(f"R_SUB = {R_SUB * 100:.2f} cm", xy=(w.x + R_SUB, w.y), xytext=(w.x + 0.12 * lim, w.y - 0.25 * lim),
                arrowprops=dict(arrowstyle="->", color=PALETTE[1]), color=PALETTE[1], fontsize=10)
    a2.set_xlim(-lim, lim)
    a2.set_ylim(-lim, lim)
    a2.set_aspect("equal")
    a2.set_xlabel("x [m]")
    a2.set_title(f"Haz de {N_SUB} subconductores, a escala")
    a2.grid(alpha=0.25)
    fig.tight_layout()
    return fig

draw_parameters();

## 3. Gradiente en la superficie y margen contra Peek

`solve` aplica el método de **simulación de cargas**: reemplaza cada conductor por varias cargas lineales ficticias dentro de él (`N_CHARGES` por conductor) y calcula su valor para que la superficie quede a la tensión del conductor, con los cables de guarda a 0 V si `GUARD_MODE = "grounded"`. Con esas cargas se calcula el campo y, de ahí, el gradiente en la superficie de cada subconductor: `E_max (simulación de cargas)`. `E_c` no sale de la simulación sino de la **fórmula de Peek**. El criterio de diseño es `E_max / E_c < 0,95`.

Los gradientes calculados con la simulación de cargas son **valores pico**. La columna rms divide entre √2 y sirve para comparar con reglas prácticas expresadas en rms, como los 17 kV/cm de algunas guías.

In [ ]:
wires, pot = build_wires()
sol = solve(wires, pot, n_charges=N_CHARGES)

delta = corona.air_density(ALTITUDE_M, TEMP_C)
e_c = corona.peek_critical_gradient(R_SUB * 100, M_SURFACE, delta)      # peak [kV/cm]; radius in cm

emax = {g: v * V_PER_M_TO_KV_PER_CM for g, v in sol.max_gradient_by_group().items()}
rows = []
for g in range(len(PHASES)):
    ratio, ok = corona.gradient_margin(emax[g], e_c)
    rows.append({"phase": "abc"[g], "E_max pico (simulación) [kV/cm]": emax[g], "E_max rms (simulación) [kV/cm]": corona.rms_from_peak(emax[g]),
                 "E_max / E_c (simulación y Peek)": ratio, "cumple < 0,95": ok})
table = pd.DataFrame(rows).set_index("phase")

print(f"Densidad relativa del aire: {delta:.3f}    E_c con la fórmula de Peek: {e_c:.2f} kV/cm pico ({corona.rms_from_peak(e_c):.2f} rms)")
print(f"0,95 * E_c: {0.95 * e_c:.2f} kV/cm pico ({0.95 * corona.rms_from_peak(e_c):.2f} rms)")
print(f"Error de la simulación de cargas en la superficie: {sol.bc_error:.1e}   (debe ser mucho menor que 1e-2)")
if GUARD_MODE == "insulated":
    print("El guarda aislado flota a [kV pico]:", [round(abs(sol.group_potentials[3 + j]) / 1e3, 1) for j in range(len(GUARDS))])
table.round(3)

## 4. Recorrido por la superficie de un subconductor

El gradiente no es igual en todos los puntos de la superficie. Aquí se recorre un subconductor del haz y se ve, punto por punto, qué valor tiene el campo.

- **A la izquierda**, el haz de la fase `ZOOM_PHASE` en el instante en que su gradiente es máximo. El color es |E| calculado con la simulación de cargas. El círculo azul marca el subconductor que se recorre (`TOUR_SUB`). El punto blanco es la posición actual sobre su superficie y la línea roja sale hacia afuera con un largo proporcional a |E|.
- **A la derecha**, |E| en la superficie de ese subconductor según la posición (0° es el punto de la derecha, 90° el de arriba). Las curvas grises son los otros subconductores del haz. El punto rojo sigue la posición actual. Las líneas negras son `E_c` y `0,95 E_c`.

Usa la barra de la animación para moverte punto por punto, o el botón de reproducir para recorrer toda la superficie. Cada cuadro avanza 5°.

Lo que se debe ver: el campo es más alto en la cara del subconductor que mira hacia afuera del haz y más bajo en la cara que mira hacia el centro, donde el campo de los subconductores vecinos se resta. La fórmula de Kuffel (sección 9), `E_max = E_prom [1 + (n − 1) r / R]`, estima el valor del punto más alto de esta curva sin hacer la simulación.

In [ ]:
# Instante en que la fase del zoom llega a su gradiente máximo, y campo normal en la superficie en ese instante.
wts, e_t = sol.max_gradient_vs_time(n_times=120)
wt_pk = wts[e_t[ZOOM_PHASE].argmax()]

bundle = [w for w in wires if w.group == ZOOM_PHASE]
sub = bundle[TOUR_SUB % len(bundle)]
En = np.abs(sol.surface_gradient_at(wt_pk, 360)) * V_PER_M_TO_KV_PER_CM      # kV/cm, one value per degree
En_sub = En[wires.index(sub)]
rb = corona.bundle_radius(N_SUB, SPACING)
hw = rb + 6 * R_SUB if N_SUB > 1 else 8 * R_SUB                              # half width of the close-up window [m]

fig, (ax_z, ax_g) = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={"width_ratios": [1.1, 1]})
cs = draw_bundle_zoom(ax_z, sol, ZOOM_PHASE, wt_pk, half_width=hw, e_max_kv_cm=emax[ZOOM_PHASE], streamlines=False)
fig.colorbar(cs, ax=ax_z, label="|E| [kV/cm] (simulación de cargas)", shrink=0.75)
ax_z.add_patch(plt.Circle((sub.x, sub.y), sub.r, fill=False, ec="tab:blue", lw=2.5, zorder=6))
ax_z.set_title(f"Fase {'abc'[ZOOM_PHASE]}, wt = {np.degrees(wt_pk):.0f} grados: subconductor {TOUR_SUB} (círculo azul)")
point, = ax_z.plot([], [], "o", ms=9, mfc="white", mec="k", zorder=8)
arrow, = ax_z.plot([], [], "-", color="tab:red", lw=3, zorder=7)

for w in bundle:
    ax_g.plot(range(360), En[wires.index(w)], color="0.78", lw=1)
ax_g.plot(range(360), En_sub, color="tab:blue", lw=2.5, label=f"subconductor {TOUR_SUB}")
ax_g.axhline(e_c, color="k", ls="--", label="E_c (fórmula de Peek)")
ax_g.axhline(0.95 * e_c, color="k", ls=":", label="0.95 E_c")
dot, = ax_g.plot([], [], "o", ms=11, color="tab:red", zorder=6)
readout = ax_g.text(0.03, 0.05, "", transform=ax_g.transAxes, fontsize=12, bbox=dict(fc="white", ec="0.6"))
ax_g.set_xlim(0, 360)
ax_g.set_xticks(range(0, 361, 45))
ax_g.set_ylim(max(0, En_sub.min() - 2), max(e_c, En_sub.max()) * 1.05)
ax_g.set_xlabel("Posición sobre la superficie [grados]  (0 = derecha, 90 = arriba)")
ax_g.set_ylabel("|E| en la superficie [kV/cm] (simulación de cargas)")
ax_g.set_title("Campo en la superficie del subconductor elegido")
ax_g.legend(loc="upper right")
ax_g.grid(alpha=0.3)
fig.tight_layout()

angles = np.arange(0, 360, 5)
reach = 0.35 * hw


def update_tour(i):
    d = int(angles[i])
    px, py = sub.x + sub.r * np.cos(np.radians(d)), sub.y + sub.r * np.sin(np.radians(d))
    length = reach * En_sub[d] / En_sub.max()
    point.set_data([px], [py])
    arrow.set_data([px, px + length * np.cos(np.radians(d))], [py, py + length * np.sin(np.radians(d))])
    dot.set_data([d], [En_sub[d]])
    readout.set_text(f"{d:3d} grados:  |E| = {En_sub[d]:5.2f} kV/cm  ({100 * En_sub[d] / e_c:3.0f} % de E_c)")


anim_tour = animation.FuncAnimation(fig, update_tour, frames=len(angles), interval=100)
plt.close(fig)

# Hacia dónde mira el punto más alto: ángulo de la línea que va del centro del haz al subconductor.
cx, cy = np.mean([w.x for w in bundle]), np.mean([w.y for w in bundle])
outward = np.degrees(np.arctan2(sub.y - cy, sub.x - cx)) % 360 if N_SUB > 1 else float("nan")
print(f"Punto más alto de este subconductor (simulación de cargas): {En_sub.max():.2f} kV/cm a {En_sub.argmax()} grados; "
      f"la dirección hacia afuera del haz es {outward:.0f} grados.")
display(HTML(anim_tour.to_jshtml()))

## 5. Gradiente en el tiempo

La curva muestra el gradiente máximo sobre la superficie de cada fase a lo largo de un ciclo de 60 Hz. Cada fase llega a su máximo en un instante distinto, separados unos 120° (la fase central no sigue exactamente ese patrón porque recibe el campo de las dos vecinas).

El gradiente alcanza su valor absoluto máximo **dos veces por ciclo**, una con cada polaridad. La ionización se repite con esa frecuencia, 120 Hz, y por eso el zumbido de la corona tiene esa frecuencia fundamental.

In [ ]:
# Fasores de tensión y formas de onda de las tres fases (se usan en las secciones 5 a 7).
pot_phasors = three_phase_potentials(V_LL_KV, PHASE_SEQUENCE)
wt_deg = np.linspace(0, 360, 361)
v_t = {k: np.real(pot_phasors[k] * np.exp(1j * np.radians(wt_deg))) / 1e3 for k in range(3)}   # kV

wts, e_t = sol.max_gradient_vs_time(n_times=120)
fig, ax = plt.subplots(figsize=(11, 4))
for k in range(3):
    ax.plot(np.degrees(wts), e_t[k] * V_PER_M_TO_KV_PER_CM, color=PALETTE[k], lw=2, label=f"fase {'abc'[k]}")
ax.axhline(e_c, color="k", ls="--", label="E_c (fórmula de Peek)")
ax.axhline(0.95 * e_c, color="k", ls=":", label="0.95 E_c")
ax.set_xlim(0, 360)
ax.set_xticks(range(0, 361, 60))
ax.set_xlabel("Ángulo eléctrico wt [grados]")
ax.set_ylabel("Máx. |E| en la superficie [kV/cm] (simulación de cargas)")
ax.set_title("Gradiente en la superficie durante un ciclo (el |E| máximo ocurre dos veces por ciclo)")
ax.legend(ncol=5, loc="lower center")
ax.grid(alpha=0.3)
for k in range(3):
    print(f"fase {'abc'[k]}: máximo {e_t[k].max() * V_PER_M_TO_KV_PER_CM:.2f} kV/cm (simulación de cargas) a wt = {np.degrees(wts[e_t[k].argmax()]):.0f} grados "
          f"(y {(np.degrees(wts[e_t[k].argmax()]) + 180) % 360:.0f} grados)")

## 6. Acercamiento a un haz: campo y líneas equipotenciales

La figura muestra, en un instante del ciclo (`ZOOM_WT_DEG`), el campo alrededor de los subconductores de la fase `ZOOM_PHASE`.

- **Color:** módulo de |E| en kV/cm, calculado con la simulación de cargas. Es más alto sobre la cara exterior de cada subconductor y baja hacia el centro del haz, donde el campo de los subconductores vecinos se cancela.
- **Líneas negras:** equipotenciales, espaciadas en pasos iguales de tensión. Se aprietan donde el campo es alto. Cerca del haz rodean al conjunto como si fuera un solo conductor de radio mayor, y cerca de cada subconductor se cierran alrededor de él.
- **Líneas azules (panel derecho):** líneas de campo. Salen del conductor en ángulo recto, como exige la condición de frontera de un conductor.

Cambia `ZOOM_PHASE` para ver otra fase y `ZOOM_WT_DEG` para ver otro instante. Con la secuencia `"abc"` los picos positivos de a, b y c ocurren en 0°, 120° y 240°.

In [ ]:
zoom_wt = np.radians(ZOOM_WT_DEG)
v_zoom = np.real(pot_phasors[ZOOM_PHASE] * np.exp(1j * zoom_wt)) / 1e3
rb = corona.bundle_radius(N_SUB, SPACING)
e_zoom_max = emax[ZOOM_PHASE]                                       # kV/cm, colour scale

fig, axs = plt.subplots(1, 2, figsize=(14, 6.2))
cs = draw_bundle_zoom(axs[0], sol, ZOOM_PHASE, zoom_wt, e_max_kv_cm=e_zoom_max, streamlines=False)
axs[0].set_title(f"|E| y equipotenciales (simulación de cargas), fase {'abc'[ZOOM_PHASE]} en wt = {ZOOM_WT_DEG:.0f} grados (V = {v_zoom:.0f} kV)")
fig.colorbar(cs, ax=axs[0], label="|E| [kV/cm] (simulación de cargas)", shrink=0.8)
draw_bundle_zoom(axs[1], sol, ZOOM_PHASE, zoom_wt, half_width=rb + 6 * R_SUB, e_max_kv_cm=e_zoom_max)
axs[1].set_title("Más cerca: líneas de campo (azul) y equipotenciales (negro)")
fig.tight_layout();

## 7. Animación de un ciclo de 60 Hz

Dos animaciones de un ciclo completo (`N_FRAMES` cuadros entre 0° y 360°):

1. **Toda la línea:** |E| instantáneo. Se ve cómo el campo se concentra alrededor de la fase que está cerca de su pico y cómo la zona de campo alto pasa de una fase a otra en el orden de la secuencia. Abajo, las tres tensiones con un cursor.
2. **Un haz:** |E|, líneas equipotenciales y líneas de campo alrededor de los subconductores de `ZOOM_PHASE`. Cuando la carga de la fase pasa por cero, el gradiente en su superficie se anula y cambia de signo.

Los botones de la animación permiten pausarla, avanzarla cuadro a cuadro y cambiar la velocidad. Para guardarla como GIF, asigna un nombre de archivo a `SAVE_GIF`.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 80          # MB allowed for the embedded frames
frames_deg = np.linspace(0, 360, N_FRAMES, endpoint=False)
xmax = max(abs(p[0]) for p in PHASES + GUARDS) + 15
ymax = max(p[1] for p in PHASES + GUARDS) + 10


def waveform_axes(ax):
    for k in range(3):
        ax.plot(wt_deg, v_t[k], color=PALETTE[k], lw=1.8, label=f"fase {'abc'[k]}")
    ax.set_xlim(0, 360)
    ax.set_xticks(range(0, 361, 60))
    ax.set_xlabel("wt [grados]")
    ax.set_ylabel("V [kV]")
    ax.grid(alpha=0.3)
    return ax.axvline(0, color="k", lw=1.5)


# ---- Animation 1: whole line ---------------------------------------------------------------------
fig1 = plt.figure(figsize=(9, 7.2))
gs1 = fig1.add_gridspec(2, 1, height_ratios=[3, 1])
ax_map, ax_wave = fig1.add_subplot(gs1[0]), fig1.add_subplot(gs1[1])
cursor1 = waveform_axes(ax_wave)
fig1.tight_layout()


def update_line(i):
    d = frames_deg[i]
    ax_map.clear()
    draw_field_map_at(ax_map, sol, np.radians(d), (-xmax, xmax), (0, ymax), vmax_kv_m=0.08 * V_LL_KV)
    ax_map.set_title(f"|E| (simulación de cargas) [kV/m]   wt = {d:5.1f} grados   t = {d / 360 / FREQ_HZ * 1e3:5.2f} ms")
    cursor1.set_xdata([d, d])


anim_line = animation.FuncAnimation(fig1, update_line, frames=N_FRAMES, interval=120)
plt.close(fig1)
display(HTML(anim_line.to_jshtml()))

In [ ]:
# ---- Animation 2: one bundle ----------------------------------------------------------------------
fig2 = plt.figure(figsize=(11, 5.6))
gs2 = fig2.add_gridspec(1, 2, width_ratios=[1.3, 1])
ax_b, ax_w2 = fig2.add_subplot(gs2[0]), fig2.add_subplot(gs2[1])
cursor2 = waveform_axes(ax_w2)
fig2.tight_layout()


def update_bundle(i):
    d = frames_deg[i]
    ax_b.clear()
    draw_bundle_zoom(ax_b, sol, ZOOM_PHASE, np.radians(d), e_max_kv_cm=emax[ZOOM_PHASE])
    v = np.real(pot_phasors[ZOOM_PHASE] * np.exp(1j * np.radians(d))) / 1e3
    ax_b.set_title(f"Fase {'abc'[ZOOM_PHASE]} (simulación de cargas): wt = {d:5.1f} grados, V = {v:6.1f} kV")
    cursor2.set_xdata([d, d])


anim_bundle = animation.FuncAnimation(fig2, update_bundle, frames=N_FRAMES, interval=120)
plt.close(fig2)
if SAVE_GIF:
    anim_bundle.save(SAVE_GIF, writer="pillow", fps=8)
    print("GIF written to", SAVE_GIF)
display(HTML(anim_bundle.to_jshtml()))

## 8. Campo eléctrico al nivel del suelo

Valor eficaz de la componente vertical del campo a 1 m de altura, a lo ancho de la línea, calculado con la simulación de cargas. Es el dato que se compara con los límites de exposición de las personas. El cable de guarda a tierra lo reduce un poco; cambia `GUARD_MODE` para verlo.

In [ ]:
xg = np.linspace(-(max(abs(p[0]) for p in PHASES + GUARDS) + 15), max(abs(p[0]) for p in PHASES + GUARDS) + 15, 241)
_, ey = sol.field(xg, np.full_like(xg, 1.0))
ground_rms = np.abs(ey) / np.sqrt(2) / 1e3                       # kV/m rms

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(xg, ground_rms, color=PALETTE[1], lw=2)
for k, (x, y) in enumerate(PHASES):
    ax.axvline(x, color=PALETTE[k], ls=":", lw=1)
ax.set_xlabel("x [m]  (líneas punteadas: posición de las fases a, b, c)")
ax.set_ylabel("|E_y| rms a 1 m [kV/m] (simulación de cargas)")
ax.set_title("Campo eléctrico al nivel del suelo (simulación de cargas)")
ax.grid(alpha=0.3)
print(f"Máximo a 1 m: {ground_rms.max():.2f} kV/m rms en x = {xg[ground_rms.argmax()]:.1f} m")

## 9. Comparación con las ecuaciones de CIGRE y Kuffel

Se compara el gradiente máximo de la fase central calculado con la **simulación de cargas** contra el que dan tres **fórmulas** de Kuffel y CIGRE, y el gradiente crítico de Peek contra otras expresiones. En cada tabla se indica qué sale de la simulación y qué sale de una fórmula.

**Gradiente máximo (CIGRE TB 638 sec. 2.6; Kuffel ec. 2.9).** Para un haz de `n` subconductores con la carga `q` de la fase,

$$E_{prom} = \frac{q}{2\pi\varepsilon_0\, n\, r}, \qquad E_{max} = E_{prom}\left[1 + (n-1)\frac{r}{R}\right], \qquad R = \frac{a}{2\sin(\pi/n)}$$

La ecuación ignora el campo de las otras fases. La simulación de cargas sí lo incluye, por eso la diferencia mide cuánto importa esa simplificación.

**Gradiente crítico.** La forma de Peek usada en el código, y la otra expresión que trae el resumen del curso, que aparece allí sin fuente clara. Si difieren, hay que revisar cuál corresponde a Cardona Correa (2022), ec. 4.

In [ ]:
xy = np.array(PHASES + (GUARDS if GUARD_MODE == "grounded" else []), float)
r_eq = [corona.equivalent_radius(N_SUB, R_SUB, SPACING)] * 3 + [R_GUARD] * (len(xy) - 3)
v = np.r_[three_phase_potentials(V_LL_KV, PHASE_SEQUENCE), np.zeros(len(xy) - 3)]
q_maxwell = corona.phase_charges(xy, r_eq, v)
q_sim = charge_per_group(sol)

b = 1                                                    # phase b (centre)
kv = lambda x: x * V_PER_M_TO_KV_PER_CM
rows = [
    ("Simulación de cargas (cálculo numérico, geometría exacta)", emax[b]),
    ("Fórmula de Kuffel / CIGRE 2.6-5, con la carga de la simulación de cargas", kv(corona.bundle_max_gradient_v_m(q_sim[b], N_SUB, R_SUB, SPACING))),
    ("Fórmula de Kuffel / CIGRE 2.6-5, con la carga de la matriz de Maxwell (sin simulación)", kv(corona.bundle_max_gradient_v_m(q_maxwell[b], N_SUB, R_SUB, SPACING))),
    ("Fórmula del gradiente medio, CIGRE 2.6-4, con la carga de la matriz de Maxwell", kv(corona.bundle_average_gradient_v_m(q_maxwell[b], N_SUB, R_SUB))),
]
cmp_emax = pd.DataFrame(rows, columns=["método (fase b)", "pico [kV/cm]"])
cmp_emax["diferencia respecto de la simulación de cargas [%]"] = 100 * (cmp_emax["pico [kV/cm]"] / emax[b] - 1)
display(cmp_emax.round(3))

r_cm = R_SUB * 100
e0_alt = 30 * np.sqrt(2) * M_SURFACE * (1 - 0.07 * r_cm) * delta ** (2 / 3)      # alternative expression quoted in the course summary
cmp_ec = pd.DataFrame([
    ("Fórmula de Peek, cable trenzado: 30 m d (1 + 0,308 / sqrt(d r))  [la usada aquí]", e_c),
    ("Fórmula de Peek, cilindro liso (constante 0,301)", corona.peek_critical_gradient(r_cm, M_SURFACE, delta, stranded=False)),
    ("Expresión alterna del resumen: 30 sqrt(2) m (1 - 0,07 r) d^(2/3)  [fuente por verificar]", e0_alt),
], columns=["fórmula del gradiente crítico", "E_c pico [kV/cm]"])
cmp_ec["diferencia respecto de la usada [%]"] = 100 * (cmp_ec["E_c pico [kV/cm]"] / e_c - 1)
display(cmp_ec.round(3))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(cmp_emax["método (fase b)"][::-1], cmp_emax["pico [kV/cm]"][::-1], color=PALETTE[1])
ax.axvline(e_c, color="k", ls="--", label="E_c (fórmula de Peek)")
ax.axvline(0.95 * e_c, color="k", ls=":", label="0.95 E_c")
ax.set_xlabel("kV/cm pico")
ax.legend(loc="lower right")
ax.set_title("Gradiente máximo de la fase b según el método")
fig.tight_layout()

### Cable de guarda: ecuaciones 4.217 a 4.220 del libro

Si el guarda está a tierra, su tensión es cero y su carga sale de las capacitancias mutuas con los conductores de fase, `Q_sw = Σ C_sw,i V_i` (ecs. 4.217 y 4.219). El gradiente medio en su superficie es `E_sw = Q_sw / (2 π ε r)` (ec. 4.220). La tabla compara esa expresión, con la carga de la matriz de Maxwell, contra el máximo de la simulación de cargas, y contra el `E_c` de Peek para el radio del guarda.

In [ ]:
# Gradiente del cable de guarda: ecuación 4.220 del libro contra la simulación de cargas.
if GUARD_MODE == "grounded" and len(GUARDS) > 0:
    q_guard = q_maxwell[3:]                                                       # C/m, from the Maxwell matrix
    e_sw = np.abs(q_guard) * corona.K_COULOMB / R_GUARD * V_PER_M_TO_KV_PER_CM    # eq. 4.220, kV/cm peak
    ec_guard = corona.peek_critical_gradient(R_GUARD * 100, M_SURFACE, delta)
    guard_table = pd.DataFrame({
        "E_sw, fórmula 4.220 [kV/cm pico]": e_sw,
        "máximo con simulación de cargas [kV/cm pico]": [emax[3 + j] for j in range(len(GUARDS))],
        "E_c del cable de guarda, fórmula de Peek [kV/cm pico]": ec_guard,
    }, index=[f"G{j + 1}" for j in range(len(GUARDS))])
    display(guard_table.round(2))
else:
    print("Set GUARD_MODE = 'grounded' and at least one shield wire to see this comparison.")

## 10. Comparación con el libro de CIGRE (Nolasco et al.)

El capítulo de diseño eléctrico de *Overhead Lines* (CIGRE Green Books, Springer, 2017), sección 4.19, resuelve un caso base: línea bipolar de ±500 kV en corriente continua, 3 conductores Lapwing por polo (diámetro 3,822 cm), separación entre subconductores 45 cm, separación entre polos 13,0 m y altura mínima 12,5 m. Esta sección repite ese ejemplo con las funciones de `corona.py` (fórmulas del libro) y con la simulación de cargas, y compara cada resultado con el valor impreso en el libro. La columna *método* dice cuál de los dos produjo cada fila.

**Qué aplica a una línea de corriente alterna y qué no.** Las ecuaciones 4.208 a 4.211 (gradiente del haz) y 4.221 (Peek) son electrostáticas y valen para corriente alterna con valores pico. Las de pérdidas, ruido audible y radiointerferencia (4.223 a 4.226) son fórmulas empíricas para líneas bipolares de corriente continua y **no se pueden aplicar a una línea trifásica**.

In [ ]:
# Caso base del libro (sección 4.19): ±500 kV bipolar, 3 x Lapwing. Alturas y distancias en cm, como en las ecuaciones.
BOOK = dict(v_kv=500.0, n=3, r_cm=1.911, a_cm=45.0, h_cm=1250.0, s_cm=1300.0)
BOOK_M, BOOK_DELTA = 0.82, 0.915            # surface factor and relative air density of the example
BOOK_ALTITUDE_M = 600.0                     # altitude used in the RI and AN examples

R_cm = corona.bundle_radius(BOOK["n"], BOOK["a_cm"])
req_cm = corona.equivalent_radius(BOOK["n"], BOOK["r_cm"], BOOK["a_cm"])
e_avg = corona.markt_mengele_average_gradient_kv_cm(**BOOK)
e_max_book = corona.bundle_max_from_average(e_avg, BOOK["n"], BOOK["r_cm"], BOOK["a_cm"])
ec_book = corona.peek_critical_gradient(BOOK["r_cm"], BOOK_M, BOOK_DELTA, stranded=False)   # the book uses 0.301
d_cm_book = 2 * BOOK["r_cm"]
loss_fair = corona.hvdc_bipolar_corona_loss_w_m(e_max_book, d_cm_book, 3, 12.5, 13.0, "fair")
loss_foul = corona.hvdc_bipolar_corona_loss_w_m(e_max_book, d_cm_book, 3, 12.5, 13.0, "foul")
radial = np.hypot(12.5, 30.0 - 6.5)           # distance from the positive pole to a point 30 m from the line centre
an_book = corona.audible_noise_dba(e_max_book, 3, d_cm_book, radial, BOOK_ALTITUDE_M)
ri_radial = corona.radio_interference_db(e_max_book, d_cm_book, radial, 1.0, BOOK_ALTITUDE_M)
ri_printed = corona.radio_interference_db(e_max_book, d_cm_book, 30.0, 1.0, BOOK_ALTITUDE_M)

# La misma geometría con la simulación de cargas: polos a x = ±6,5 m y 12,5 m de altura, +500 kV y -500 kV.
wires_b = []
add_bundle(wires_b, -6.5, 12.5, 3, 0.45, 0.01911, group=0, label="+")
add_bundle(wires_b, 6.5, 12.5, 3, 0.45, 0.01911, group=1, label="-")
sol_b = solve(wires_b, {0: 500e3, 1: -500e3}, n_charges=16)
q_b = abs(charge_per_group(sol_b)[0])
e_avg_sim = corona.bundle_average_gradient_v_m(q_b, 3, 0.01911) * V_PER_M_TO_KV_PER_CM
e_max_sim = sol_b.max_gradient_by_group()[0] * V_PER_M_TO_KV_PER_CM

F, S = "fórmula del libro", "simulación de cargas"
rows = [
    ("R [cm]", "4.212", 26.0, R_cm, F),
    ("r_eq [cm]", "4.213", 15.7, req_cm, F),
    ("E_a, gradiente medio [kV/cm]", "4.214", 20.297, e_avg, F),
    ("E_m, gradiente máximo [kV/cm]", "4.215", 23.28, e_max_book, F),
    ("E_c, gradiente crítico (Peek) [kV/cm]", "4.221", 27.6, ec_book, F),
    ("Pérdidas por corona, buen tiempo [W/m]", "4.223", 3.7, loss_fair, F),
    ("Pérdidas por corona, mal tiempo [W/m]", "4.224", 20.6, loss_foul, F),
    ("Pérdidas por corona, 80 % buen tiempo + 20 % mal tiempo [W/m]", "-", 7.1, 0.8 * loss_fair + 0.2 * loss_foul, F),
    ("Ruido audible [dBA]", "4.226", 38.2, an_book, F),
    ("Radiointerferencia, D = distancia radial 26,6 m [dB]", "4.225", 41.8, ri_radial, F),
    ("Radiointerferencia, D = 30 m como está impreso [dB]", "4.225", 41.8, ri_printed, F),
    ("E_a con simulación de cargas [kV/cm]", "4.214", 20.297, e_avg_sim, S),
    ("E_m con simulación de cargas [kV/cm]", "4.215", 23.28, e_max_sim, S),
]
book_table = pd.DataFrame(rows, columns=["cantidad", "ecuación del libro", "valor del libro", "este notebook", "método"])
book_table["diferencia [%]"] = 100 * (book_table["este notebook"] / book_table["valor del libro"] - 1)
book_table.round(3)

**Cómo leer la tabla.**

- Las ecuaciones del libro se reproducen: radio del haz, radio equivalente, gradiente medio y máximo, gradiente crítico, pérdidas y ruido audible coinciden con los valores impresos dentro del redondeo del libro.
- **Radiointerferencia.** El libro escribe `19,9 / 30` en la ecuación, pero el valor que imprime (41,8 dB) sale con la distancia radial al polo positivo, 26,6 m, la misma que usa en el ejemplo de ruido audible. Con 30 m el resultado es 39,7 dB.
- **Constante de Peek.** El libro usa 0,301 (cilindro liso) con `m` = 0,82. En el resto del notebook se usa 0,308 (cable trenzado), que da `E_c` un 0,4 % mayor.
- **Simulación de cargas contra fórmulas.** El gradiente medio de la simulación coincide con la ecuación 4.214. El máximo sale algo mayor que el de la ecuación 4.215 porque la simulación incluye el otro polo y el plano de tierra con la geometría exacta. El libro dice que las ecuaciones son razonablemente exactas para `n ≤ 4`.

### Figura 4.62 del libro: gradiente contra radio del subconductor

La figura 4.62 del libro muestra el gradiente máximo en la superficie en función del radio del subconductor para 1 a 4 conductores por polo, con `E_c` y `0,95 E_c`. Aquí las líneas son las ecuaciones 4.208 y 4.209, los círculos son la simulación de cargas y las dos líneas horizontales gruesas son `E_c` y `0,95 E_c` con `m` = 0,82 y δ = 0,915. Cambia `BOOK_SPACING_CM` para ver el efecto de la separación del haz.

Para comparar con el libro, ponlo al lado de la figura 4.62: los valores se leen a ojo con una incertidumbre de unos 0,5 kV/cm. El libro no da la geometría que usó para dibujarla, así que pueden quedar diferencias de uno o dos kV/cm para radios pequeños.

In [ ]:
BOOK_SPACING_CM = 45.0        # bundle spacing used in the figure [cm]; the book's base case uses 45
radii = np.linspace(1.0, 2.4, 15)
colors = {1: "tab:blue", 2: "tab:purple", 3: "tab:green", 4: "tab:red"}

fig, ax = plt.subplots(figsize=(9, 5.6))
cmp_rows = []
for n_ in (1, 2, 3, 4):
    eq = []
    for r_ in radii:
        ea_ = corona.markt_mengele_average_gradient_kv_cm(500.0, n_, r_, 1250.0, 1300.0, BOOK_SPACING_CM)
        eq.append(corona.bundle_max_from_average(ea_, n_, r_, BOOK_SPACING_CM))
    ax.plot(radii, eq, color=colors[n_], lw=1.6, label=f"{n_} cond.: fórmulas 4.208 y 4.209")
    for r_ in radii[::2]:
        w_ = []
        add_bundle(w_, -6.5, 12.5, n_, BOOK_SPACING_CM / 100, r_ / 100, group=0)
        add_bundle(w_, 6.5, 12.5, n_, BOOK_SPACING_CM / 100, r_ / 100, group=1)
        s_ = solve(w_, {0: 500e3, 1: -500e3}, n_charges=16)
        e_sim_ = s_.max_gradient_by_group()[0] * V_PER_M_TO_KV_PER_CM
        ax.plot(r_, e_sim_, "o", color=colors[n_], mfc="none", ms=6)
        cmp_rows.append({"n": n_, "r [cm]": r_, "fórmula": eq[list(radii).index(r_)], "simulación de cargas": e_sim_})
ec_curve = np.array([corona.peek_critical_gradient(r_, BOOK_M, BOOK_DELTA, stranded=False) for r_ in radii])
ax.plot(radii, ec_curve, color="tab:orange", lw=2, label="E_c (fórmula de Peek)")
ax.plot(radii, 0.95 * ec_curve, color="k", lw=3, label="0.95 E_c")
ax.plot([], [], "o", color="0.4", mfc="none", label="simulación de cargas (círculos)")
ax.set_xlim(1.0, 2.4)
ax.set_ylim(10, 40)
ax.set_xlabel("Radio del subconductor [cm]")
ax.set_ylabel("Gradiente máximo en la superficie [kV/cm]")
ax.set_title("±500 kV bipolar: gradiente en la superficie (compare con la figura 4.62 del libro)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)

fig_cmp = pd.DataFrame(cmp_rows)
fig_cmp["diferencia [%]"] = 100 * (fig_cmp["simulación de cargas"] / fig_cmp["fórmula"] - 1)
print(f"Mayor diferencia entre las fórmulas y la simulación de cargas: {fig_cmp['diferencia [%]'].abs().max():.1f} %")
fig_cmp.pivot(index="r [cm]", columns="n", values=["fórmula", "simulación de cargas"]).round(2)

## 11. Variaciones y datos para el taller

Esta sección sirve para hacer ejercicios con otros valores sin tocar la sección 1.

- **`analyze(...)`** repite todo el cálculo con algunos parámetros cambiados, por ejemplo `analyze(SPACING=0.6, N_SUB=3)`, y devuelve el gradiente de cada fase calculado con la simulación de cargas, el `E_c` de la fórmula de Peek y el margen. Los nombres de los parámetros son los de la sección 1.
- **Barrido.** `SWEEP_PARAM` y `SWEEP_VALUES` definen el parámetro y los valores que se recorren. Con `"N_SUB"` y `[1, 2, 3, 4, 5, 6]` se ve cuándo el haz empieza a cumplir el criterio de 0,95. Prueba también `"SPACING"`, `"ALTITUDE_M"` o `"M_SURFACE"`.
- **Exportar.** `export_results()` guarda en `OUTPUT_DIR` los parámetros y los resultados en CSV, para abrirlos en Excel o entregarlos en el taller.
- **Cargar tu geometría.** `save_geometry(...)` escribe la geometría actual en un CSV con las columnas `kind` (`phase` o `guard`), `x` e `y` en metros. Edítalo y en la sección 1 descomenta la línea `PHASES, GUARDS = load_geometry("my_line.csv")`.

In [ ]:
# Esta función repite todo el cálculo con otros valores de los parámetros de la sección 1 (por nombre).
def analyze(**overrides):
    """Peak gradient of each phase, Peek limit and margin, with some parameters replaced.

    Example: ``analyze(SPACING=0.6, N_SUB=3)``. The globals are restored afterwards.
    """
    g = globals()
    saved = {k: g[k] for k in overrides}
    try:
        g.update(overrides)
        wires_, pot_ = build_wires()
        sol_ = solve(wires_, pot_, n_charges=max(N_CHARGES, 16))
        delta_ = corona.air_density(ALTITUDE_M, TEMP_C)
        ec_ = corona.peek_critical_gradient(R_SUB * 100, M_SURFACE, delta_)
        e_ = sol_.max_gradient_by_group()
        peaks = [e_[k] * V_PER_M_TO_KV_PER_CM for k in range(3)]
        worst = max(peaks)
        return {"E_max a (simulación) [kV/cm]": peaks[0], "E_max b (simulación) [kV/cm]": peaks[1],
                "E_max c (simulación) [kV/cm]": peaks[2], "E_c (Peek) [kV/cm]": ec_,
                "E_max / E_c, peor fase (simulación y Peek)": worst / ec_, "cumple < 0,95": worst / ec_ < 0.95}
    finally:
        g.update(saved)


print(analyze())

In [ ]:
# Barrido de un parámetro: nombre y lista de valores. Cambia estas dos líneas para otros ejercicios.
SWEEP_PARAM = "N_SUB"
SWEEP_VALUES = [1, 2, 3, 4, 5, 6]

sweep = pd.DataFrame([{SWEEP_PARAM: v, **analyze(**{SWEEP_PARAM: v})} for v in SWEEP_VALUES]).set_index(SWEEP_PARAM)

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ratio = sweep["E_max / E_c, peor fase (simulación y Peek)"]
ax.bar([str(i) for i in sweep.index], ratio, color=[PALETTE[2] if r < 0.95 else PALETTE[0] for r in ratio])
ax.axhline(0.95, color="k", ls="--", label="0.95")
ax.set_xlabel(SWEEP_PARAM)
ax.set_ylabel("E_max / E_c, peor fase (simulación y Peek)")
ax.set_title("Barrido (en verde, las que cumplen el criterio)")
ax.legend()
fig.tight_layout()
sweep.round(3)

In [ ]:
# Esta función guarda los resultados en archivos CSV dentro de OUTPUT_DIR para abrirlos en Excel u otro programa.
def export_results(folder=None):
    """Write the parameters and the results of this notebook as CSV/JSON files. Returns the list of files."""
    import json
    folder = pathlib.Path(folder or OUTPUT_DIR)
    folder.mkdir(parents=True, exist_ok=True)
    names = ("V_LL_KV", "PHASES", "N_SUB", "SPACING", "R_SUB", "GUARDS", "R_GUARD", "GUARD_MODE", "ALTITUDE_M",
             "TEMP_C", "M_SURFACE", "FREQ_HZ", "PHASE_SEQUENCE", "N_CHARGES")
    files = []

    def save(df, name, **kw):
        path = folder / name
        df.to_csv(path, **kw)
        files.append(path)

    (folder / "parameters.json").write_text(json.dumps({k: globals()[k] for k in names}, indent=1))
    files.append(folder / "parameters.json")

    save(table, "phase_summary.csv")                                                     # section 3
    phi, E = sol.surface_gradient(360)                                                   # peak kV/cm by angle
    count = {}
    cols = {}
    for i, w in enumerate(wires):
        count[w.group] = count.get(w.group, 0) + 1
        cols[f"{w.label}{count[w.group] - 1}_kV_per_cm"] = E[i] * V_PER_M_TO_KV_PER_CM
    save(pd.DataFrame({"angle_deg": np.degrees(phi), **cols}), "surface_gradient_peak.csv", index=False)
    wts_, e_t_ = sol.max_gradient_vs_time(n_times=120)                                   # section 5
    save(pd.DataFrame({"wt_deg": np.degrees(wts_), **{f"phase_{'abc'[k]}_kV_per_cm": e_t_[k] * V_PER_M_TO_KV_PER_CM
                                                      for k in range(3)}}), "gradient_vs_time.csv", index=False)
    save(pd.DataFrame({"x_m": xg, "E_rms_kV_per_m": ground_rms}), "ground_field.csv", index=False)   # section 8
    save(sweep, "sweep.csv")                                                             # this section
    save(book_table, "book_comparison.csv", index=False)                                 # section 10
    return files


files = export_results()
print("Files written to", pathlib.Path(OUTPUT_DIR).resolve())
for f in files:
    print("  ", f.name)
if "google.colab" in sys.modules:                       # in Google Colab the files are downloaded to your computer
    from google.colab import files as colab_files
    for f in files:
        colab_files.download(str(f))

In [ ]:
# Plantilla de geometría para editar en Excel: se escribe con los valores actuales de PHASES y GUARDS.
save_geometry(pathlib.Path(OUTPUT_DIR) / "geometry_template.csv", PHASES, GUARDS)
print(pd.read_csv(pathlib.Path(OUTPUT_DIR) / "geometry_template.csv"))

## Ejercicios adicionales

1. Cambie el número de haces de conductores y haga variación de la separación.
2. Sube `ALTITUDE_M` a 3000 y observa cuánto se acerca la fase central a `0,95 E_c`. Vuelve a bajarla y cambia `M_SURFACE` a 0,5 para simular lluvia.
3. Reduce la separación entre fases a 8 m (`PHASES`). Explica por qué cambia el gradiente de la fase central y el de las laterales.
4. Cambia `GUARD_MODE` entre `"grounded"`, `"insulated"` y `"none"`. ¿En cuál de los tres cambia el gradiente de las fases? Compara con el campo en el suelo de la sección 8.
5. Con `N_SUB = 3`, barre `SPACING` entre 0,10 y 0,60 m con la sección 11. ¿Hay una separación que cumple `E_max / E_c < 0,95`? ¿En qué separación el gradiente es mínimo y por qué sube hacia los dos lados? Si no cumple, ¿qué otro parámetro cambiaría?
6. En la sección 9, ¿qué tan lejos está la ecuación de Kuffel de la simulación cuando las fases están a 5 m una de otra? Explica la diferencia.
7. Cambia `PHASE_SEQUENCE` a `"acb"` y compara las secciones 5 y 7. ¿Qué cambia en la animación y qué no cambia en los gradientes máximos?
8. En la sección 6, elige `ZOOM_WT_DEG` igual al ángulo en que la tensión de la fase se anula y luego 90° después. Compara las equipotenciales y el gradiente en la superficie.
9. En la sección 4, recorre cada subconductor del haz (`TOUR_SUB` = 0, 1, 2) y anota el ángulo donde el gradiente es máximo. ¿Hacia dónde apunta respecto al centro del haz? ¿Cuánto vale el gradiente en la cara que mira al centro?
10. En la sección 10, reproduce la figura 4.62 con una separación del haz de 35 cm y de 60 cm (`BOOK_SPACING_CM`). ¿Qué radio mínimo cumple `0,95 E_c` con 3 subconductores?
11. Con `SWEEP_PARAM = "SPACING"` y valores de 0,30 a 0,60 m, exporta los resultados a CSV y grafica `E_max / E_c` contra la separación en Excel.
12. Escribe la geometría de otra línea en un CSV (plantilla de la sección 11), cárgala con `load_geometry` y compara su margen contra Peek.

## Fuentes

- Cardona Correa, L. (2022). *Formulación y taller para el cálculo de pérdidas por efecto corona en líneas aéreas de alta tensión*. UNAL Sede Medellín.
- CIGRE Study Committee B2, J. F. Nolasco et al. (2017). Cap. 4 «Electrical Design», sec. 4.19, ecs. 4.208 a 4.227, en *Overhead Lines*, CIGRE Green Books, Springer.
- CIGRE WG B2.40 (2016). *Guide to overall overhead line design*, Technical Brochure 638, sec. 2.6 y 5.5–5.7.
- Kuffel, E., Zaengl, W. S., Kuffel, J. *High Voltage Engineering Fundamentals*, ec. 2.9.
- Peek, F. W. (1929). *Dielectric Phenomena in High Voltage Engineering*.

El código de cálculo está en `src/electric_field/` (`corona.py`, `charge_simulation.py`) y sus pruebas en `tests/test_corona.py`, incluido el caso base del libro.